In [0]:
-- ===================================================
-- BLOCK 1 — GOVERNANCE NAMESPACE (SQL)
-- ===================================================

-- Separate security metadata and consumer-facing views from the physical
-- Bronze, Silver, and Gold data products.

CREATE SCHEMA IF NOT EXISTS semiconplus_portfolio.governance
COMMENT 'Access mappings and security-policy metadata for SemiconPlus';

CREATE SCHEMA IF NOT EXISTS semiconplus_portfolio.secure
COMMENT 'Governed views exposed to authorized analytics consumers';

In [0]:
-- ===================================================
-- BLOCK 2 — USER ENTITLEMENT TABLE (SQL)
-- ===================================================

-- Maintain product-group access independently from pipeline tables so access
-- changes do not require rewriting manufacturing data.

CREATE TABLE IF NOT EXISTS
semiconplus_portfolio.governance.user_product_group_access
(
    principal STRING NOT NULL
        COMMENT 'Databricks account identity returned by session_user()',

    product_group_id STRING NOT NULL
        COMMENT 'Product group that the principal is authorized to access',

    access_role STRING NOT NULL
        COMMENT 'Business role assigned to the principal',

    can_view_equipment_details BOOLEAN NOT NULL
        COMMENT 'Controls whether equipment identifiers remain visible',

    is_active BOOLEAN NOT NULL
        COMMENT 'Disables access without deleting entitlement history',

    granted_at_utc TIMESTAMP NOT NULL
        COMMENT 'Timestamp when the entitlement was created or updated',

    granted_by STRING NOT NULL
        COMMENT 'Identity that created or last updated the entitlement'
)
USING DELTA
COMMENT 'Product-group row entitlements and equipment-detail permissions';

In [0]:
-- ===================================================
-- BLOCK 3 — DEVELOPMENT USER ENTITLEMENTS (SQL)
-- ===================================================

-- Assign two controlled product groups to the project owner. PG01 masks
-- equipment identifiers while PG02 permits equipment-level investigation.

MERGE INTO semiconplus_portfolio.governance.user_product_group_access AS target
USING
(
    SELECT
        'bugastokenpatrick@gmail.com' AS principal,
        'PG01' AS product_group_id,
        'DEVICE_OWNER_ANALYST' AS access_role,
        FALSE AS can_view_equipment_details

    UNION ALL

    SELECT
        'bugastokenpatrick@gmail.com',
        'PG02',
        'DEVICE_OWNER_ANALYST',
        TRUE
) AS source
ON  LOWER(target.principal) = LOWER(source.principal)
AND target.product_group_id = source.product_group_id

WHEN MATCHED THEN UPDATE SET
    target.access_role = source.access_role,
    target.can_view_equipment_details =
        source.can_view_equipment_details,
    target.is_active = TRUE,
    target.granted_at_utc = CURRENT_TIMESTAMP(),
    target.granted_by = SESSION_USER()

WHEN NOT MATCHED THEN INSERT
(
    principal,
    product_group_id,
    access_role,
    can_view_equipment_details,
    is_active,
    granted_at_utc,
    granted_by
)
VALUES
(
    source.principal,
    source.product_group_id,
    source.access_role,
    source.can_view_equipment_details,
    TRUE,
    CURRENT_TIMESTAMP(),
    SESSION_USER()
);

In [0]:
-- ===================================================
-- BLOCK 4 — VERIFY SECURITY IDENTITY (SQL)
-- ===================================================

-- Confirm that the runtime identity matches an active entitlement before
-- publishing the secure device-owner view.

SELECT
    SESSION_USER() AS session_identity,
    product_group_id,
    access_role,
    can_view_equipment_details,
    is_active
FROM semiconplus_portfolio.governance.user_product_group_access
WHERE LOWER(principal) = LOWER(SESSION_USER())
ORDER BY product_group_id;

In [0]:
-- ===================================================
-- BLOCK 5 — DEVICE-OWNER DYNAMIC VIEW (SQL)
-- ===================================================

-- Enforce product-group row security through the entitlement join. Exclude
-- ingestion metadata, pseudonymize event identifiers, and conditionally mask
-- equipment identifiers without duplicating the underlying Silver data.

CREATE OR REPLACE VIEW
semiconplus_portfolio.secure.vw_device_owner_test_results
COMMENT 'Row-filtered and column-restricted test-result view for device owners'
AS
SELECT
    SHA2(
        CONCAT(
            COALESCE(results.event_id, 'UNKNOWN'),
            ':SEMICONPLUS'
        ),
        256
    ) AS event_reference,

    results.event_timestamp_utc,
    results.product_group_id,
    results.device_id,
    results.site_id,

    CASE
        WHEN access.can_view_equipment_details
            THEN results.equipment_id
        ELSE 'RESTRICTED'
    END AS equipment_id,

    results.status,
    results.test_time_seconds,

    access.access_role,
    access.can_view_equipment_details

FROM semiconplus_portfolio.silver.streaming_test_results AS results

INNER JOIN
semiconplus_portfolio.governance.user_product_group_access AS access
    ON results.product_group_id = access.product_group_id
   AND LOWER(access.principal) = LOWER(SESSION_USER())
   AND access.is_active = TRUE;

In [0]:
-- ===================================================
-- BLOCK 6 — OPERATIONS AGGREGATE VIEW (SQL)
-- ===================================================

-- Provide site and product-group performance monitoring without exposing raw
-- event identifiers, device identifiers, equipment identifiers, or lineage
-- metadata.

CREATE OR REPLACE VIEW
semiconplus_portfolio.secure.vw_operations_hourly_yield
COMMENT 'Hourly aggregate yield metrics for manufacturing operations'
AS
SELECT
    DATE_TRUNC('HOUR', event_timestamp_utc) AS reporting_hour_utc,
    site_id,
    product_group_id,

    COUNT(*) AS total_event_count,

    COUNT_IF(status = 'PASS') AS pass_count,
    COUNT_IF(status = 'FAIL') AS fail_count,
    COUNT_IF(status = 'ALARM') AS alarm_count,

    COUNT_IF(status IN ('PASS', 'FAIL')) AS tested_unit_count,

    TRY_DIVIDE(
        COUNT_IF(status = 'PASS'),
        COUNT_IF(status IN ('PASS', 'FAIL'))
    ) AS first_pass_yield,

    AVG(test_time_seconds) AS average_test_time_seconds,
    MAX(event_timestamp_utc) AS latest_event_timestamp_utc

FROM semiconplus_portfolio.silver.streaming_test_results

GROUP BY
    DATE_TRUNC('HOUR', event_timestamp_utc),
    site_id,
    product_group_id;

In [0]:
SELECT
    table_schema,
    table_name,
    table_type
FROM semiconplus_portfolio.information_schema.tables
WHERE table_schema = 'secure'
ORDER BY table_name;

In [0]:
-- ===================================================
-- BLOCK 7 — CONSUMER PRIVILEGES (SQL)
-- ===================================================

-- Grant the project owner access to the governed presentation layer. A
-- production deployment would replace this user principal with account-level
-- consumer groups managed through the Databricks account console.

GRANT USE CATALOG
ON CATALOG semiconplus_portfolio
TO `bugastokenpatrick@gmail.com`;

GRANT USE SCHEMA
ON SCHEMA semiconplus_portfolio.secure
TO `bugastokenpatrick@gmail.com`;

GRANT SELECT
ON VIEW semiconplus_portfolio.secure.vw_device_owner_test_results
TO `bugastokenpatrick@gmail.com`;

GRANT SELECT
ON VIEW semiconplus_portfolio.secure.vw_operations_hourly_yield
TO `bugastokenpatrick@gmail.com`;

In [0]:
-- ===================================================
-- BLOCK 8 — DEVICE-OWNER ROW-SECURITY VALIDATION (SQL)
-- ===================================================

-- Confirm that every visible product group is explicitly assigned to the
-- current identity and that no unauthorized product groups leak through.

WITH visible_groups AS
(
    SELECT DISTINCT product_group_id
    FROM semiconplus_portfolio.secure.vw_device_owner_test_results
),

entitled_groups AS
(
    SELECT DISTINCT product_group_id
    FROM semiconplus_portfolio.governance.user_product_group_access
    WHERE LOWER(principal) = LOWER(SESSION_USER())
      AND is_active = TRUE
)

SELECT
    visible.product_group_id AS unauthorized_product_group
FROM visible_groups AS visible

LEFT ANTI JOIN entitled_groups AS entitled
    ON visible.product_group_id = entitled.product_group_id;

In [0]:
-- ===================================================
-- BLOCK 9 — COLUMN-MASK VALIDATION (SQL)
-- ===================================================

-- Reconcile the masking outcome against each product-group entitlement.

SELECT
    product_group_id,
    can_view_equipment_details,

    COUNT(*) AS visible_rows,

    COUNT_IF(equipment_id = 'RESTRICTED')
        AS masked_equipment_rows,

    COUNT_IF(equipment_id <> 'RESTRICTED')
        AS visible_equipment_rows

FROM semiconplus_portfolio.secure.vw_device_owner_test_results

GROUP BY
    product_group_id,
    can_view_equipment_details

ORDER BY product_group_id;

In [0]:
-- ===================================================
-- BLOCK 10 — RESTRICTED-COLUMN VALIDATION (SQL)
-- ===================================================

-- Confirm that raw identifiers and technical lineage columns are absent from
-- the consumer-facing device-owner contract.

SELECT column_name
FROM semiconplus_portfolio.information_schema.columns
WHERE table_schema = 'secure'
  AND table_name = 'vw_device_owner_test_results'
  AND
  (
      column_name = 'event_id'
      OR column_name LIKE '\\_source%'
      OR column_name LIKE '\\_ingested%'
      OR column_name LIKE '\\_pipeline%'
      OR column_name = '_rescued_data'
  );

In [0]:
-- ===================================================
-- BLOCK 11 — OPERATIONS-GRAIN VALIDATION (SQL)
-- ===================================================

-- Confirm that the operations view contains one record per reporting grain and
-- that calculated yield values remain within their valid range.

SELECT
    reporting_hour_utc,
    site_id,
    product_group_id,
    COUNT(*) AS duplicate_count
FROM semiconplus_portfolio.secure.vw_operations_hourly_yield
GROUP BY
    reporting_hour_utc,
    site_id,
    product_group_id
HAVING COUNT(*) > 1;

In [0]:
-- ===================================================
-- BLOCK 12 — OPERATIONS-METRIC VALIDATION (SQL)
-- ===================================================

SELECT *
FROM semiconplus_portfolio.secure.vw_operations_hourly_yield
WHERE total_event_count <= 0
   OR pass_count < 0
   OR fail_count < 0
   OR alarm_count < 0
   OR first_pass_yield < 0
   OR first_pass_yield > 1;

In [0]:
-- ===================================================
-- BLOCK 13 — SECURITY OBJECT INVENTORY (SQL)
-- ===================================================

SELECT
    table_schema,
    table_name,
    table_type
FROM semiconplus_portfolio.information_schema.tables
WHERE table_schema IN ('governance', 'secure')
ORDER BY table_schema, table_name;

In [0]:
-- ===================================================
-- BLOCK 14 — FINAL SECURITY ACCEPTANCE (SQL)
-- ===================================================

SELECT
    SESSION_USER() AS validated_identity,

    (
        SELECT COUNT(*)
        FROM semiconplus_portfolio.secure
            .vw_device_owner_test_results
    ) AS device_owner_visible_rows,

    (
        SELECT COUNT(DISTINCT product_group_id)
        FROM semiconplus_portfolio.secure
            .vw_device_owner_test_results
    ) AS device_owner_visible_product_groups,

    (
        SELECT COUNT(*)
        FROM semiconplus_portfolio.secure
            .vw_operations_hourly_yield
    ) AS operations_aggregate_rows,

    CURRENT_TIMESTAMP() AS validated_at_utc,

    'PASSED' AS security_acceptance_status;